<img src='https://gitlab.eumetsat.int/eumetlab/oceans/ocean-training/tools/frameworks/-/raw/main/img/MSOC_banner.png' align='right' width='100%'/>

<a href="../Index.ipynb">Index</a> | <a href="./Access_PACE_EarthData.ipynb">Accessing PACE OCI through EarthData</a> | <a href="./Access_MSI_CDSE.ipynb">Accessing Sentinel-2 MSI data via the CDSE</a> | <a href="./Access_multisensor_CMEMS_Data_Store.ipynb">Accessing multisensor data through the CMEMS Data Store</a>

**Copyright:** 2026 European Union <br>
**License:** MIT <br>
**Authors:** Ben Loveday (Innoflair for EUMETSAT), Hayley Evers-King (EUMETSAT)

<div class="alert alert-block alert-success">
<h3>Multi-sensor Ocean Colour Course</h3></div>

<div class="alert alert-block alert-warning">
    
<b>PREREQUISITES </b>
    
This notebook has the following prerequisites:
- **<a href="https://user.eumetsat.int/register" target="_blank">A EUMETSAT User Portal account</a>** if you are using or plan to use the EUMETSAT Data Store

There are no prerequisite notebooks for this module.
</div>
<hr>

## Accessing Copernicus Sentinel-3 OLCI data via the EUMETSAT Data Store

### Data used

| Dataset | EUMETSAT collection ID | EUMETSAT collection<br>description | WEkEO dataset ID | WEkEO description |
|:--------------------:|:-----------------------:|:-------------:|:-----------------:|:-----------------:|
| Sentinel-3 OLCI level 2 full resolution (operational) | EO:EUM:DAT:0407 | <a href="https://user.eumetsat.int/catalogue/EO:EUM:DAT:SENTINEL-3:OL_2_WFR___NTC" target="_blank">Description</a> | EO:EUM:DAT:SENTINEL-3:OL_2_WFR___ | <a href="https://www.wekeo.eu/data?view=dataset&dataset=EO%3AEUM%3ADAT%3ASENTINEL-3%3AOL_2_WFR___" target="_blank">Description</a> |
| Sentinel-3 OLCI level 2 full resolution (version BC003) reprocessing | EO:EUM:DAT:0556 | <a href="https://user.eumetsat.int/catalogue/EO:EUM:DAT:0556" target="_blank">Description</a> | EO:EUM:DAT:SENTINEL-3:0556 | <a href="https://wekeo.copernicus.eu/data?view=dataset&dataset=EO%3AEUM%3ADAT%3ASENTINEL-3%3A0556" target="_blank">Description</a> |
| Sentinel-3 OLCI level 1b full resolution (operational) | EO:EUM:DAT:0409 | <a href="https://user.eumetsat.int/catalogue/EO:EUM:DAT:SENTINEL-3:OL_1_EFR___NTC" target="_blank">Description</a> | EO:EUM:DAT:SENTINEL-3:OL_1_EFR___ | <a href="https://www.wekeo.eu/data?view=dataset&dataset=EO%3AEUM%3ADAT%3ASENTINEL-3%3AOL_1_EFR___" target="_blank">Description</a> |
| Sentinel-3 OLCI level 1b full resolution (version BC004) reprocessing| EO:EUM:DAT:0885 | <a href="https://user.eumetsat.int/catalogue/EO:EUM:DAT:0885" target="_blank">Description</a> | EO:EUM:DAT:0885 | <a href="https://wekeo.copernicus.eu/data?view=dataset&dataset=EO%3AEUM%3ADAT%3A0885" target="_blank">Description</a> |

### Learning outcomes

At the end of this notebook you will know;
* how to search and download data from the EUMETSAT Data Store using the <font color="#138D75">**eumetsat data access (eumdac)**</font> client
* How to refine your <font color="#138D75">**searches**</font> for Copernicus Sentinel-3 OLCI marine products
* How to refine your searches for temporal, spatial, orbital and flag coverage parameters
* How to <font color="#138D75">**download**</font> full products and their components

### Outline

The EUMETSAT Data Store provides access to all Copernicus Sentinel-3 marine data, as processed by EUMETSAT. This includes the most up to date and comprehensive operational and reprocessed collections for OLCI. The Data Store offers a variety of interfaces, including a web GUI and suite of APIs to support automated, programmatic access. In this notebook, we will show you how to use these APIs to search for, filter and download OLCI data, by exploiting the `eumdac` <a href="https://user.eumetsat.int/resources/user-guides/eumetsat-data-access-client-eumdac-guide" target="_blank">toolkit</a>.

<hr>

<div class="alert alert-info" role="alert">

## Importing dependencies

</div>

We begin by importing all of the libraries that we need to run this notebook. If you have built your python using the environment file provided in this repository, then you should have everything you need. For more information on building environment, please see the repository **<a href="../README.md" target="_blank">README</a>**.

In [1]:
import datetime               # a library that supports the creation of date objects
import os                     # a library that allows us access to basic operating system commands like making directories
import shutil                 # a library that allows us access to basic operating system commands like copy
import zipfile                # a library that allows us to unzip zip-files.
import eumdac                 # a tool that helps us download via the eumetsat/data-store
from pathlib import Path      # a library that helps construct system path objects
import getpass                # a library to help us enter passwords
import eumartools             # a EUMETSAT library that support working with Copernicus Sentinel products

Next we will create a download directory to store the products we will download in this notebook.

In [2]:
download_dir = os.path.join(os.path.dirname(os.getcwd()), "Data", "Preprepared", "Day1", "Data_access", "OLCI")
os.makedirs(download_dir, exist_ok=True)

<div class="alert alert-info" role="alert">

##  Downloading from the Data Store via API

</div>

<div class="alert alert-block alert-success">

### Accessing the EUMETSAT Data Store

To download Copernicus marine products from the <a href="https://data.eumetsat.int" target="_blank">EUMETSAT Data Store</a>, we will use the EUMETSAT Data Access Client (`eumdac`) python package. If you are working with the recommended Anaconda Python distribution and used the environment file included in this repository (environment.yml) to build this python environment (as detailed in the README), you will already have installed this. If not, you can install eumdac using;

`conda install -c eumetsat eumdac`

You can also find the source code on the <a href="https://gitlab.eumetsat.int/eumetlab/data-services/eumdac " target="_blank">EUMETSAT GitLab</a>. Please visit the EUMETSAT user portal for more information on the <a href="https://user.eumetsat.int/data-access/data-store " target="_blank">EUMETSAT Data Store</a> and <a href="https://user.eumetsat.int/resources/user-guides/eumetsat-data-access-client-eumdac-guide " target="_blank">eumdac</a>.

To download data from the EUMETSDAT Data Store via API, you need to provide credentials. To obtain these you should first register at for an <a href="https://user.eumetsat.int/register" target="_blank">EUMETSAT User Portal</a> account. To access Copernicus data, you must log in to your User Portal account, open your profile by clicking on your name, navigate to the ***"My data licenses"*** tab and tick the ***"Meteosat > 1 hr latency & Metop, Copernicus data & Third party data"*** box, then log out. It can take up to 1 hour for this change to take effect. Failure to complete these steps will result in 503 authentication errors when downloading Copernicus products. Existing users with Earth Observation Portal accounts have already been migrated to the User Portal, and do not need to reconfigure their licenses. 

Once you have an account and the requisite licenses, you can retrieve your `<your_consumer_key>` and `<your_consumer_secret>` from the <a href="https://api.eumetsat.int/api-key/" target="_blank">EUMETSAT Data Store API</a> page (*Note: you must click the "Show hidden fields" button at the bottom of the page to see the relevant fields*). If you do not already have a local credentials file, you will be prompted to enter your credentials when you run the cell below. This will create the required local credentials file, so that you only need to run this once.

*Note: your key and secret are permanent, so you should take care to never share them*

</div>

Lets create our credentials file, if it doesn't already exist, and read our credentials from it.

In [3]:
# load credentials
eumdac_credentials_file = Path(Path.home() / '.eumdac' / 'credentials')

if os.path.exists(eumdac_credentials_file):
    consumer_key, consumer_secret = Path(eumdac_credentials_file).read_text().split(',')
else:
    # creating authentication file
    consumer_key = input('Enter your consumer key: ')
    consumer_secret = getpass.getpass('Enter your consumer secret: ')
    try:
        os.makedirs(os.path.dirname(eumdac_credentials_file), exist_ok=True)
        with open(eumdac_credentials_file, "w") as f:
            f.write(f'{consumer_key},{consumer_secret}')
    except:
        pass

Now we will use these credentials to create a token to interface with the EUMETSAT Data Store...

In [4]:
token = eumdac.AccessToken((consumer_key, consumer_secret))
print(f"This token '{token}' expires {token.expiration}")

This token '09e59236-a909-3e17-9a69-fa23e82052ab' expires 2026-04-09 12:57:29.769078


Now we have a token, we can create an instance of the EUMETSAT Data Store.

In [5]:
datastore = eumdac.DataStore(token)

Data in the EUMETSAT Data Store are stored as collections, each with its own ID. If we don't know our collection ID a priori, we can find this information via the Data Store GUI or by asking `eumdac` to tell us about all available relevant collections as follows:

In [6]:
for collection_id in datastore.collections:
    if ("OLCI" in collection_id.title) and not "non-public" in collection_id.abstract:
        print(f"Collection ID({collection_id}): {collection_id.title}")

Collection ID(EO:EUM:DAT:0557): OLCI Level 2 Ocean Colour Reduced Resolution (version BC003) - Sentinel-3 - Reprocessed
Collection ID(EO:EUM:DAT:0556): OLCI Level 2 Ocean Colour Full Resolution (version BC003) - Sentinel-3 - Reprocessed
Collection ID(EO:EUM:DAT:0885): OLCI Level 1B Full Resolution (version BC004) - Sentinel-3 - Reprocessed
Collection ID(EO:EUM:DAT:0886): OLCI Level 1B Reduced Resolution (version BC004) - Sentinel-3 - Reprocessed
Collection ID(EO:EUM:DAT:0407): OLCI Level 2 Ocean Colour Full Resolution - Sentinel-3
Collection ID(EO:EUM:DAT:0409): OLCI Level 1B Full Resolution - Sentinel-3
Collection ID(EO:EUM:DAT:0408): OLCI Level 2 Ocean Colour Reduced Resolution - Sentinel-3
Collection ID(EO:EUM:DAT:0410): OLCI Level 1B Reduced Resolution - Sentinel-3


So, for;
* **OLCI Level 1B Full Resolution** we want `collection_id`: **EO:EUM:DAT:0409** (or **EO:EUM:DAT:0885** for the latest reprocessing)
* **OLCI Level 2 Ocean Colour Full Resolution** we want `collection_id`: **EO:EUM:DAT:0407** (or **EO:EUM:DAT:0556** for the latest reprocessing)

Lets get some level-1B data first. Below we provide the level-1B `collection_id` to our `datastore` object to choose the correct collection.

In [7]:
collection_id = 'EO:EUM:DAT:0407'
selected_collection = datastore.get_collection(collection_id)

If we happen to know the exact id/name of our product (e.g. from the <a href="https://data.eumetsat.int" target="_blank">EUMETSAT Data Store GUI</a>), we can use it directly as follows;

`product_id = 'S3B_OL_1_EFR____20250403T101615_20250403T101915_20250404T133909_0179_105_065_1980_MAR_O_NT_004.SEN3'`

`selected_product = datastore.get_product(product_id=product_id, collection_id=collection_id)`

However, we are going to assume that we don't know this information, and want to search instead. We have multiple options for filtering, which we can query as follows:

In [8]:
selected_collection.search_options

{'bbox': {'title': 'Inventory which has a spatial extent overlapping this bounding box',
  'options': []},
 'geo': {'title': 'Inventory which has a spatial extent overlapping this Well Known Text geometry',
  'options': []},
 'title': {'title': 'Can be used to define a wildcard search on the product title (product identifier), use set notation as OR and space as AND operator between multiple search terms',
  'options': [None]},
 'sat': {'title': 'Mission / Satellite',
  'options': ['Sentinel-3B', 'Sentinel-3A']},
 'type': {'title': 'Product Type', 'options': ['OL_2_WFR___']},
 'dtstart': {'title': 'Temporal Start', 'options': []},
 'dtend': {'title': 'Temporal End', 'options': []},
 'publication': {'title': 'publication date', 'options': []},
 'zone': {'title': 'Equi7grid main continental zone',
  'options': ['NA', 'OC', 'AS', 'AN', 'SA', 'AF', 'EU']},
 't6': {'title': 'Equi7grid 600km tile', 'options': []},
 'timeliness': {'title': 'Timeliness', 'options': ['NT', 'NR']},
 'orbit': {'t

Typically, we want to search by time, space, mission, and timeliness. We'll configure all of these as options below. Lets start by setting some time bounds. We can do this using Python datetime objects, as follow;

In [9]:
# time filter the collection for matching products
dtstart = datetime.datetime(2025, 8, 13, 0, 0)
dtend = datetime.datetime(2025, 8, 14, 0, 0)

Next, lets decide on the timeliness of the data. If we are working with very recent data, then perhaps we want to use near real-time (NR) products. If not, we want to use non-time critical products (NT). When working with reprocessed collections, we always want to use NT.

In [10]:
timeliness = "NT"

Now lets determine a spatial region. We can do this in one of two ways, using either:

* the `bbox` option, which defines a rectangular polygon using [west, south, east, north], or
* the `geo` option, which defines a polygon of any shape using a WKT format, e.g. 'POLYGON ((10.09 56.09, 10.34 56.09, 10.34 56.19, 10.09 56.19, 10.09 56.09))'

Lets define a regular box....

In [11]:
west = 2.0
south = 54.0
east = 3.0
north = 55.0

The options below would yield identical results:

In [12]:
bbox = f"{west}, {south}, {east}, {north}"
roi = [[west, south], [west, north], [east, north], [east, south], [west, south]]
geo = 'POLYGON(({}))'.format(','.join(["{} {}".format(*coord) for coord in roi]))

print(f"bbox: {bbox}")
print(f"geo: {geo}")

bbox: 2.0, 54.0, 3.0, 55.0
geo: POLYGON((2.0 54.0,2.0 55.0,3.0 55.0,3.0 54.0,2.0 54.0))


It is important to note that three thing about this search:

1. it will return any granules that **intersect** between the satellite granule, so there is currently no guarantee that the granule will entirely cover your ROI.
2. as level-1B and level-2 products are on the instrument grid, the granules will not be clipped to the ROI.
3. the is no guarantee that the data over your ROI will be good quality; it may well be completely cloudy.

With a little bit of clever filtering, we can address the first and last points. Lets define an `overlap_threshold` of 1.0, mandating that our downloaded granules completely cover our ROI. Lets also set a `flag_coverage_threshold` that allows us to pre-screen our products for flag coverage **within** the ROI.

In [13]:
overlap_threshold = 1.0
flag_coverage_threshold = 0.2

If we provide information about the flag file that corresponds to the product type, as below, we can use methods in the `eumartools` package to "pre-screen" products prior to full download. Note that this is done through local testing, and not remotely, but only small files are downloaded.

In [14]:
if "OLL1" in selected_collection.product_type:
    flag_var = "quality_flags"
    flag_file = "qualityFlags.nc"
    flags = ["bright"]
else:
    flag_var = "WQSF"
    flag_file = "wqsf.nc"
    flags = ["CLOUD"]

Now we have defined everything, lets check how many products correspond to our search **before** pre-screening.

In [15]:
products = list(selected_collection.search(
    geo=geo,
    timeliness=timeliness,
    dtstart=dtstart, 
    dtend=dtend))

for product, num in zip(products,range(1,len(products)+1)):
    print(f"{str(num).zfill(2)}: {product}")

01: S3A_OL_2_WFR____20250813T103242_20250813T103542_20250814T183344_0180_129_165_1980_MAR_O_NT_003.SEN3
02: S3B_OL_2_WFR____20250813T095402_20250813T095702_20250814T161934_0179_110_022_1980_MAR_O_NT_003.SEN3


Now we can iterate through all these products, ignoring the ones that do not correspond to our "pre-screen" criteria (done by the `check_polygon_overlap` and `check_OLCI_flag_coverage` methods);

In [16]:
for product in products:
    overlap = eumartools.check_polygon_overlap(product, roi, download_dir)
    flag_coverage = eumartools.check_OLCI_flag_coverage(product, roi, download_dir, flags=flags, flag_var=flag_var, flag_file=flag_file)

    if overlap < overlap_threshold or flag_coverage > flag_coverage_threshold:
        print(f"❌ Skipping product: {product} (overlap: {str(overlap)}, flag coverage: {str(flag_coverage)}")
        continue

    with product.open() as fsrc, open(os.path.join(download_dir, fsrc.name), mode='wb') as fdst:
        print(f'Downloading {fsrc.name}')
        shutil.copyfileobj(fsrc, fdst)
        print(f'✅ Download complete: {fsrc.name}.')

    with zipfile.ZipFile(fdst.name, 'r') as zip_ref:
        for file in zip_ref.namelist():
            if file.startswith(str(product)):
                zip_ref.extract(file, download_dir)
        print(f'✅ Unzip complete: {fdst.name}')

    os.remove(fdst.name)

✅ Download complete: S3A_OL_2_WFR____20250813T103242_20250813T103542_20250814T183344_0180_129_165_1980_MAR_O_NT_003.SEN3.zip.
✅ Unzip complete: /Users/benloveday/Code/git_repositories/CMTS/internal/ocean/applications/ocean-colour-applications/3_courses/msoc/Data/Preprepared/Day1/Data_access/OLCI/S3A_OL_2_WFR____20250813T103242_20250813T103542_20250814T183344_0180_129_165_1980_MAR_O_NT_003.SEN3.zip
✅ Download complete: S3B_OL_2_WFR____20250813T095402_20250813T095702_20250814T161934_0179_110_022_1980_MAR_O_NT_003.SEN3.zip.
✅ Unzip complete: /Users/benloveday/Code/git_repositories/CMTS/internal/ocean/applications/ocean-colour-applications/3_courses/msoc/Data/Preprepared/Day1/Data_access/OLCI/S3B_OL_2_WFR____20250813T095402_20250813T095702_20250814T161934_0179_110_022_1980_MAR_O_NT_003.SEN3.zip


Hopefully, that worked successfully and you should have downloaded two level-2 OLCI products.

<div class="alert alert-success" role="alert">

## What next?

You should be able to adapt the methods above to search the EUMETSAT Data Store for the Copernicus Sentinel-3 OLCI level-1B and level-2 data you require for this course, using your project ROI and time bounds a as input. If you need more information on using `eumdac` to interface with the EUMETSAT Data Store you can find a comprehensive overview in the <a href="https://user.eumetsat.int/resources/user-guides/eumetsat-data-access-client-eumdac-guide#ID-Usage-examples" target="_blank">online manual</a>.

Remember; you will need to adapt the `collection` to search for OLCI level-1b products. If you are using this notebook as the basis for gathering data beyond this course, you should pay particular attention to the differing collections for reprocessed products, and product timeliness. 

</div>

<hr>
<a href="../Index.ipynb">Index</a> | <a href="./Access_PACE_EarthData.ipynb">Accessing PACE OCI through EarthData</a> | <a href="./Access_MSI_CDSE.ipynb">Accessing Sentinel-2 MSI data via the CDSE</a> | <a href="./Access_multisensor_CMEMS_Data_Store.ipynb">Accessing multisensor data through the CMEMS Data Store</a>
<hr>
<a href="https://gitlab.eumetsat.int/eumetlab/ocean">View on GitLab</a> | <a href="https://training.eumetsat.int/">EUMETSAT Training</a> | <a href=mailto:ops@eumetsat.int>Contact helpdesk for support </a> | <a href=mailto:.training@eumetsat.int>Contact our training team to collaborate on and reuse this material</a></span></p>